# 04 · TF-IDF + Ridge hyperparameter sweeps (Days 8-10)

Three sequential development-set sweeps to fix the baseline TF-IDF + Ridge
configuration for Empathy prediction. Each sweep holds the other
hyperparameters fixed so its effect can be isolated.

1. **Day 8 — n-gram range.** Unigrams vs unigrams+bigrams.
2. **Day 9 — vocabulary size.** `max_features` ∈ {1000, 5000, 10000, 20000}.
3. **Day 10 — regularization strength.** Ridge α ∈ {0.1, 1, 3, 10, 100}.

**Final configuration** (justified by the sweeps below):
`TfidfVectorizer(ngram_range=(1,1), max_features=10000)` with
`Ridge(alpha=3.0)`, trained on the training split and evaluated on both dev
and test.

**Split note.** This notebook uses the same internal conversation-grouped
70/15/15 split as `02_baseline_multi_target.ipynb`, kept for consistency
with the exploratory experiments in that notebook. All hyperparameter
decisions are made on dev; test is used only for final reporting.


## 0 · Setup


In [1]:
!git clone https://github.com/DavorSopar/dataset-analysis.git /content/dataset-analysis
import sys
sys.path.insert(0, '/content/dataset-analysis/src')

Cloning into '/content/dataset-analysis'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 35 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 1.41 MiB | 4.24 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [8]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

# Make src/ importable whether run from notebooks/ or the repo root.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").is_dir():
    pass
elif (REPO_ROOT.parent / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
from data import load_convt, impute_selfdisclosure, TARGETS

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

In [3]:
# Load and resolve the two missing SelfDisclosure values (speaker-mean; see 01_eda).
df = impute_selfdisclosure(load_convt())
print(f"{len(df):,} turns across {df['conversation_id'].nunique()} conversations")
print("Targets:", TARGETS)

11,166 turns across 487 conversations
Targets: ['Emotion', 'EmotionalPolarity', 'Empathy', 'SelfDisclosure']


### Split, features, metric helpers

Same conversation-grouped split, TF-IDF setup, and metric helpers as
`02_baseline_multi_target.ipynb`. Reproduced here so this notebook can be
run standalone.


In [4]:
groups = df["conversation_id"].to_numpy()

# 70% train, 30% temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(gss1.split(df, groups=groups))

# split temp 50/50 -> 15% dev, 15% test (still grouped)
temp = df.iloc[temp_idx]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
dev_rel, test_rel = next(gss2.split(temp, groups=temp["conversation_id"].to_numpy()))
dev_idx, test_idx = temp_idx[dev_rel], temp_idx[test_rel]

train, dev, test = df.iloc[train_idx], df.iloc[dev_idx], df.iloc[test_idx]

# Guarantee no conversation appears in more than one split.
assert not (set(train.conversation_id) & set(dev.conversation_id))
assert not (set(train.conversation_id) & set(test.conversation_id))
assert not (set(dev.conversation_id) & set(test.conversation_id))

split_tbl = pd.DataFrame({
    "turns": [len(train), len(dev), len(test)],
    "conversations": [train.conversation_id.nunique(),
                      dev.conversation_id.nunique(),
                      test.conversation_id.nunique()],
}, index=["train", "dev", "test"])
split_tbl["turns_%"] = (100 * split_tbl["turns"] / len(df)).round(1)
print(split_tbl)
print("\nNo conversation overlaps across splits. Good.")

       turns  conversations  turns_%
train   7788            340  69.7000
dev     1640             73  14.7000
test    1738             74  15.6000

No conversation overlaps across splits. Good.


In [5]:
def pearson(y_true, y_pred):
    """Pearson r; returns NaN when either side is constant (undefined)."""
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        return np.nan
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def score_all(y_true, y_pred):
    return {
        "pearson": pearson(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }

In [6]:
# Target for this notebook: Empathy only.
TARGET = 'Empathy'
y_train = train[TARGET]
y_dev = dev[TARGET]
y_test = test[TARGET]


## 1 · Day 8 — n-gram range

**Goal.** Test whether phrase-level features (bigrams) capture empathy
signal beyond individual words. Unigrams alone treat `"not sorry"` as
the same as `"sorry not"`; adding bigrams preserves short phrases that
encode negation, intensity, and idiom.

**Setup.** Two `TfidfVectorizer` configurations, everything else fixed
(`max_features=5000`, `Ridge(alpha=3.0)`, dev evaluation).


In [9]:
# Or i can just simplify everything and do it with a for loop
configs = [
    {"name": "unigrams",         "ngram_range": (1, 1)},
    {"name": "unigrams+bigrams", "ngram_range": (1, 2)},
]
results = []
for cfg in configs:
    vec = TfidfVectorizer(ngram_range=cfg["ngram_range"], max_features=5000)
    X_tr = vec.fit_transform(train["text"])
    X_dv = vec.transform(dev["text"])
    model = Ridge(alpha=3.0).fit(X_tr, y_train)
    preds = model.predict(X_dv)
    mae = mean_absolute_error(y_dev, preds)
    rmse = np.sqrt(mean_squared_error(y_dev, preds))
    r = pearsonr(y_dev, preds)[0]
    results.append({"name": cfg["name"], "mae": mae, "rmse": rmse, "pearson": r})
    print(f"{cfg['name']:20s} MAE={mae:.4f} RMSE={rmse:.4f} Pearson={r:.4f}")

unigrams             MAE=0.5643 RMSE=0.7086 Pearson=0.6405
unigrams+bigrams     MAE=0.5655 RMSE=0.7104 Pearson=0.6396


**Result.** Unigrams and unigrams+bigrams produced essentially identical
dev performance (Pearson 0.6405 vs 0.6396) — a difference well within
noise. Phrase-level features do not add meaningful signal for empathy
prediction on this data at this feature dimensionality. Unigrams retained
for baseline simplicity.


## 2 · Day 9 — vocabulary size (`max_features`)

**Goal.** Establish the vocabulary size where performance saturates. Too
few features drops informative words (underfitting); too many either adds
rare noisy features or is bounded by the corpus's actual vocabulary size.

**Setup.** Winning ngram range from Day 8 (`(1,1)`), fixed `alpha=3.0`,
sweep `max_features ∈ {1000, 5000, 10000, 20000}` on dev.


In [10]:
# Using the winning ngram_range from Day 8, train Ridge with max_features = 1000, 5000, 10000, 20000.
configs1 = [1000, 5000, 10000, 20000]

results=[]
for cfg in configs1:
  vec1 = TfidfVectorizer(ngram_range=(1,1), max_features=cfg)
  X_tr = vec1.fit_transform(train["text"])
  X_dv = vec1.transform(dev["text"])
  model = Ridge(alpha=3.0).fit(X_tr, y_train)
  preds = model.predict(X_dv)
  mae = mean_absolute_error(y_dev, preds)
  rmse = np.sqrt(mean_squared_error(y_dev, preds))
  r = pearsonr(y_dev, preds)[0]
  results.append({"size": cfg, "mae": mae, "rmse": rmse, "pearson": r})
  print(f"Vocabulary size={cfg} MAE={mae:.4f} RMSE={rmse:.4f} Pearson={r:.4f}")

Vocabulary size=1000 MAE=0.5686 RMSE=0.7147 Pearson=0.6325
Vocabulary size=5000 MAE=0.5643 RMSE=0.7086 Pearson=0.6405
Vocabulary size=10000 MAE=0.5632 RMSE=0.7068 Pearson=0.6425
Vocabulary size=20000 MAE=0.5632 RMSE=0.7068 Pearson=0.6425


In [11]:
print(f"Vocabulary size requested: {cfg}, actual: {len(vec1.get_feature_names_out())}")

Vocabulary size requested: 20000, actual: 8190


**Result.** Dev Pearson improved from 0.6325 at 1000 features to 0.6425
at 10000, then plateaued. The plateau reflects a natural ceiling: the
training corpus contains only 8,190 unique unigrams, so any
`max_features` value above that threshold produces identical vocabulary
and identical results. `max_features=10000` selected — it uses the full
available vocabulary without imposing an artificially small cap.


## 3 · Day 10 — regularization strength (α)

**Goal.** Confirm the optimal Ridge regularization strength for the final
configuration. Small α weakens the L2 penalty and lets coefficients grow
large, allowing the model to fit training-specific noise (overfitting).
Large α forces coefficients toward zero regardless of the data, preventing
the model from capturing real patterns (underfitting).

**Setup.** Winning ngram and vocabulary size from Days 8-9, sweep
`α ∈ {0.1, 1, 3, 10, 100}` on dev.


In [12]:
alphas = [0.1, 1.0, 3.0, 10.0, 100.0]

vec = TfidfVectorizer(ngram_range=(1,1), max_features=10000)
X_tr = vec.fit_transform(train["text"])
X_dv = vec.transform(dev["text"])

for alpha in alphas:
    model = Ridge(alpha=alpha).fit(X_tr, y_train)
    preds = model.predict(X_dv)
    mae = mean_absolute_error(y_dev, preds)
    rmse = np.sqrt(mean_squared_error(y_dev, preds))
    r = pearsonr(y_dev, preds)[0]
    print(f"alpha={alpha:>6}: MAE={mae:.4f} RMSE={rmse:.4f} Pearson={r:.4f}")

alpha=   0.1: MAE=0.6239 RMSE=0.7956 Pearson=0.5651
alpha=   1.0: MAE=0.5646 RMSE=0.7118 Pearson=0.6358
alpha=   3.0: MAE=0.5632 RMSE=0.7068 Pearson=0.6425
alpha=  10.0: MAE=0.5805 RMSE=0.7229 Pearson=0.6294
alpha= 100.0: MAE=0.6619 RMSE=0.8042 Pearson=0.5572


**Result.** Dev Pearson traced a clean U-shaped bias-variance curve:
0.5651 at α=0.1, rising to a peak of 0.6425 at α=3.0, then falling to
0.5572 at α=100. Both extremes clearly underperform, confirming the
expected bias-variance tradeoff. **α=3.0 selected as final regularization
strength.**


## 4 · Final Ridge baseline

Retrain on the training split with the fully-justified configuration and
evaluate on both dev and test. These are the canonical Ridge baseline
numbers cited throughout the thesis.

**Configuration:** `TfidfVectorizer(ngram_range=(1,1), max_features=10000)`
with `Ridge(alpha=3.0)`, target = Empathy.


In [13]:
# Final config from Days 8-10 sweeps
BEST_ALPHA = 3.0

# Rebuild vectorizer and transform all three splits
vec = TfidfVectorizer(ngram_range=(1,1), max_features=10000)
X_tr = vec.fit_transform(train["text"])
X_dv = vec.transform(dev["text"])
X_te = vec.transform(test["text"])

y_train = train["Empathy"]
y_dev = dev["Empathy"]
y_test = test["Empathy"]

# Train the final Ridge model
final_ridge = Ridge(alpha=BEST_ALPHA).fit(X_tr, y_train)

# Evaluation helper
def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r = pearsonr(y_true, y_pred)[0]
    print(f"{label:35s}  MAE={mae:.4f}  RMSE={rmse:.4f}  Pearson={r:.4f}")
    return {"label": label, "mae": mae, "rmse": rmse, "pearson": r}

# Evaluate on both splits
dev_results = evaluate(y_dev, final_ridge.predict(X_dv), "Final Ridge baseline · dev")
test_results = evaluate(y_test, final_ridge.predict(X_te), "Final Ridge baseline · test")

Final Ridge baseline · dev           MAE=0.5632  RMSE=0.7068  Pearson=0.6425
Final Ridge baseline · test          MAE=0.6060  RMSE=0.7573  Pearson=0.6429


### Persist the final model and result table

Saved to `results/` and `models/` so downstream notebooks and the thesis
write-up can reference these numbers and reload the model without
retraining.


In [14]:
import joblib

MODELS_DIR = REPO_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# Save the metrics table
pd.DataFrame([dev_results, test_results]).to_csv(
    RESULTS_DIR / 'final_ridge_baseline_results.csv', index=False
)

# Save the trained model and its vectorizer
joblib.dump(final_ridge, MODELS_DIR / 'final_ridge_baseline.pkl')
joblib.dump(vec,         MODELS_DIR / 'final_ridge_vectorizer.pkl')

print('Saved results/final_ridge_baseline_results.csv')
print('Saved models/final_ridge_baseline.pkl')
print('Saved models/final_ridge_vectorizer.pkl')


Saved results/final_ridge_baseline_results.csv
Saved models/final_ridge_baseline.pkl
Saved models/final_ridge_vectorizer.pkl


## Summary

Three independent development-set sweeps fixed the final Ridge baseline
configuration:

| Sweep | Range tested | Selected |
|---|---|---|
| N-gram range | (1,1), (1,2) | (1,1) |
| max_features | 1000, 5000, 10000, 20000 | 10000 |
| Ridge α | 0.1, 1, 3, 10, 100 | 3.0 |

Final baseline: `TfidfVectorizer(ngram_range=(1,1), max_features=10000)`
with `Ridge(alpha=3.0)`. Numbers reported in `results/final_ridge_baseline_results.csv`
and cited in the thesis Results section.
